In [1]:
%load_ext autoreload
%autoreload 2

## Table creations statements

In [2]:
districts_stmt = """CREATE TABLE districts (
    name VARCHAR(100) PRIMARY KEY,
    geom GEOMETRY(MultiPolygon, 4326)
);
"""

In [3]:
neighborhoods_stmt = """
CREATE TABLE neighborhoods (
    id INTEGER PRIMARY KEY,
    district_name VARCHAR(100) REFERENCES districts(name),
    population INTEGER,
    income FLOAT,
    name VARCHAR(100) NOT NULL,
    geom GEOMETRY(MultiPolygon, 4326)
);
"""

In [4]:
stations_stmt = """
CREATE TABLE stations (
    id INTEGER PRIMARY KEY,
    neighborhood_id INTEGER REFERENCES neighborhoods(id),
    street VARCHAR(255),
    capacity INTEGER,
    geom GEOMETRY(Point, 4326)
);
"""

In [5]:
station_status_stmt = """
CREATE TABLE station_status_input (
    station_id INTEGER REFERENCES stations(id),
    report_time TIMESTAMPTZ,
    num_bikes INTEGER,
    num_ebikes INTEGER,
    num_mechanical INTEGER,
    docks_free INTEGER,
    status VARCHAR(20),
    PRIMARY KEY (station_id, report_time)
);
"""

In [6]:
station_status_mdb_stmt = """
CREATE TABLE station_status_mdb (
    station_id INTEGER PRIMARY KEY REFERENCES stations(id),
    bikes_history tint,
    ebikes_history tint,
    mechanical_history tint,
    docks_history tint,
    status_history ttext
);"""

## Data Ingestion

In [7]:
import pandas as pd

In [8]:
from src.utils.database_connection import DatabaseConnector

database = DatabaseConnector(
    '.env'
)

## Create tables

In [9]:
database.execute_query(districts_stmt)

Database connection established successfully.


In [10]:
database.execute_query(neighborhoods_stmt)

In [11]:
database.execute_query(stations_stmt)
database.execute_query(station_status_stmt)
database.execute_query(station_status_mdb_stmt)

### Districts

In [12]:
districts = pd.read_parquet(r"data/districts.parquet", engine='pyarrow')

In [13]:
districts.head()

,district_name,geometry
0,Ciutat Vella,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x1f\x03...
1,Eixample,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\xb6\x00...
2,Sants-Montjuïc,b'\x01\x06\x00\x00\x00\x02\x00\x00\x00\x01\x03...
3,Les Corts,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00|\x02\x0...
4,Sarrià-Sant Gervasi,b'\x01\x06\x00\x00\x00\x02\x00\x00\x00\x01\x03...


In [14]:
insert_query = """
INSERT INTO districts (name, geom) 
VALUES (
    %s, 
    ST_Transform(
        ST_Multi(ST_SetSRID(ST_GeomFromWKB(%s), 25831)), 
        4326
    )
)
ON CONFLICT (name) DO UPDATE SET geom = EXCLUDED.geom;
"""

In [15]:
for index, row in districts.iterrows():
    try:
        database.execute_query(insert_query, params=(row['district_name'], row['geometry']), fetch=False)
        print(f"Successfully ingested: {row['district_name']}")
    except Exception as e:
        print(f"Failed to ingest {row['district_name']}: {e}")

Successfully ingested: Ciutat Vella
Successfully ingested: Eixample
Successfully ingested: Sants-Montjuïc
Successfully ingested: Les Corts
Successfully ingested: Sarrià-Sant Gervasi
Successfully ingested: Gràcia
Successfully ingested: Horta-Guinardó
Successfully ingested: Nou Barris
Successfully ingested: Sant Andreu
Successfully ingested: Sant Martí


### Neighbourhoods

In [16]:
neighs = pd.read_parquet(r"data/neighborhoods/neighborhoods.parquet", engine='pyarrow')

In [17]:
neighs.head()

,codi_districte,nom_districte,codi_barri,nom_barri,geometry,income_2022,population_total
0,1,Ciutat Vella,1,el Raval,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00R\x00\x0...,60.94,49917
1,1,Ciutat Vella,2,el Barri Gòtic,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00b\x00\x0...,83.07,27878
2,1,Ciutat Vella,3,la Barceloneta,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\xd2\x01...,78.43,14749
3,1,Ciutat Vella,4,"Sant Pere, Santa Caterina i la Ribera",b'\x01\x03\x00\x00\x00\x01\x00\x00\x002\x00\x0...,85.45,22767
4,2,Eixample,5,el Fort Pienc,b'\x01\x03\x00\x00\x00\x01\x00\x00\x001\x00\x0...,105.50,36621


In [18]:
import json


with open(r"data/neighborhoods-poligons.json", 'r', encoding='utf-8') as f:
    neighs_poly = json.load(f)

In [19]:
insert_query = """
    INSERT INTO neighborhoods (id, district_name, population, income, name, geom) 
    VALUES (
        %s, %s, %s, %s, %s, 
        ST_Transform(ST_Multi(ST_SetSRID(ST_GeomFromWKB(%s), 25831)), 4326)
    )
    ON CONFLICT (id) DO UPDATE SET 
        geom = EXCLUDED.geom,
        population = EXCLUDED.population,
        income = EXCLUDED.income;
    """

In [20]:
for item in neighs_poly:
    try:
        # 1. Parse geometry from ETRS89 (UTM)
        # If item['geometria_etrs89'] is a string (WKT), use wkt.loads
        # If it's a dict (GeoJSON style), use shape()
        from shapely import wkt
        geom_obj = wkt.loads(item['geometria_etrs89'])
        geom_wkb = geom_obj.wkb  # Convert to binary for PostGIS
        
        params = (
            int(item['codi_barri']),
            item['nom_districte'],
            int(item.get('population_total', 0)),
            float(item.get('income_2022', 0)),
            item['nom_barri'],
            geom_wkb
        )
        
        database.execute_query(insert_query, params=params, fetch=False)
        print(f"Successfully ingested: {item['nom_barri']}")
        
    except Exception as e:
        print(f"Failed to ingest {item.get('nom_barri', 'Unknown')}: {e}")

Successfully ingested: el Raval
Successfully ingested: el Barri Gòtic
Successfully ingested: la Barceloneta
Successfully ingested: Sant Pere, Santa Caterina i la Ribera
Successfully ingested: el Fort Pienc
Successfully ingested: la Sagrada Família
Successfully ingested: la Dreta de l'Eixample
Successfully ingested: l'Antiga Esquerra de l'Eixample
Successfully ingested: la Nova Esquerra de l'Eixample
Successfully ingested: Sant Antoni
Successfully ingested: el Poble-sec
Successfully ingested: la Marina del Prat Vermell
Successfully ingested: la Marina de Port
Successfully ingested: la Font de la Guatlla
Successfully ingested: Hostafrancs
Successfully ingested: la Bordeta
Successfully ingested: Sants - Badal
Successfully ingested: Sants
Successfully ingested: les Corts
Successfully ingested: la Maternitat i Sant Ramon
Successfully ingested: Pedralbes
Successfully ingested: Vallvidrera, el Tibidabo i les Planes
Successfully ingested: Sarrià
Successfully ingested: les Tres Torres
Successfu

In [21]:
update_query = """
    UPDATE neighborhoods 
    SET 
        population = %s,
        income = %s
    WHERE id = %s;
    """
    
database.connect()
for index, item in neighs.iterrows():
    # Data cleaning: ensure types are correct for PostgreSQL
    try:
        neighborhood_id = int(item['codi_barri'])
        population = int(item.get('population_total', 0))
        income = float(item.get('income_2022', 0))
        
        params = (population, income, neighborhood_id)
        database.execute_query(update_query, params=params, fetch=False)
        print(f"Updated stats for neighborhood ID: {neighborhood_id}")
        
    except Exception as e:
        print(f"Failed to update ID {item.get('codi_barri')}: {e}")

database.disconnect()

Updated stats for neighborhood ID: 1
Updated stats for neighborhood ID: 2
Updated stats for neighborhood ID: 3
Updated stats for neighborhood ID: 4
Updated stats for neighborhood ID: 5
Updated stats for neighborhood ID: 6
Updated stats for neighborhood ID: 7
Updated stats for neighborhood ID: 8
Updated stats for neighborhood ID: 9
Updated stats for neighborhood ID: 10
Updated stats for neighborhood ID: 11
Updated stats for neighborhood ID: 12
Updated stats for neighborhood ID: 13
Updated stats for neighborhood ID: 14
Updated stats for neighborhood ID: 15
Updated stats for neighborhood ID: 16
Updated stats for neighborhood ID: 17
Updated stats for neighborhood ID: 18
Updated stats for neighborhood ID: 19
Updated stats for neighborhood ID: 20
Updated stats for neighborhood ID: 21
Updated stats for neighborhood ID: 22
Updated stats for neighborhood ID: 23
Updated stats for neighborhood ID: 24
Updated stats for neighborhood ID: 25
Updated stats for neighborhood ID: 26
Updated stats for nei

### Stations

In [22]:
stations = pd.read_parquet(f"data/bicing_station_locations.parquet", engine='pyarrow')

In [23]:
stations.head()

,station_id,name,lat,lon,capacity
0,264,"C/ FERRAN JUNOY, 10",41.438189,2.195388,33
1,289,"C/ MÚRCIA, 64",41.416698,2.190995,27
2,536,Estación de TESTING (no usuarios),41.347695,2.119480,2
3,1,"GRAN VIA CORTS CATALANES, 760",41.397978,2.180107,46
4,2,"C/ ROGER DE FLOR, 126",41.395488,2.177198,29


In [24]:
insert_station_query = """
INSERT INTO stations (id, neighborhood_id, street, capacity, geom) 
SELECT 
    %s, 
    n.id, 
    %s, 
    %s, 
    ST_SetSRID(ST_Point(%s, %s), 4326)
FROM neighborhoods n
WHERE ST_Within(ST_SetSRID(ST_Point(%s, %s), 4326), n.geom)
ON CONFLICT (id) DO UPDATE SET 
    neighborhood_id = EXCLUDED.neighborhood_id,
    street = EXCLUDED.street,
    capacity = EXCLUDED.capacity,
    geom = EXCLUDED.geom;
"""

In [25]:
for index, row in stations.iterrows():
    # coordinates in head: lat, lon
    lat = float(row['lat'])
    lon = float(row['lon'])
    
    params = (
        int(row['station_id']),
        row['name'],
        int(row['capacity']),
        lon, lat, # ST_Point(x, y) -> lon, lat
        lon, lat  # For the subquery ST_Within
    )
    
    try:
        database.execute_query(insert_station_query, params=params, fetch=False)
        print(f"Successfully ingested station: {row['station_id']}")
    except Exception as e:
        print(f"Failed to ingest station {row['station_id']}: {e}")

Database connection established successfully.
Successfully ingested station: 264
Successfully ingested station: 289
Successfully ingested station: 536
Successfully ingested station: 1
Successfully ingested station: 2
Successfully ingested station: 3
Successfully ingested station: 4
Successfully ingested station: 5
Successfully ingested station: 6
Successfully ingested station: 7
Successfully ingested station: 8
Successfully ingested station: 9
Successfully ingested station: 10
Successfully ingested station: 11
Successfully ingested station: 12
Successfully ingested station: 13
Successfully ingested station: 14
Successfully ingested station: 15
Successfully ingested station: 17
Successfully ingested station: 18
Successfully ingested station: 19
Successfully ingested station: 20
Successfully ingested station: 21
Successfully ingested station: 22
Successfully ingested station: 23
Successfully ingested station: 24
Successfully ingested station: 25
Successfully ingested station: 26
Successf

### Station Updates

In [26]:
example_registry = pd.read_parquet(f"data/bicing_stations_status/2025_01_Gener_BicingNou_ESTACIONS_filtered.parquet", engine='pyarrow')

In [27]:
example_registry.head()

,station_id,num_bikes_available,num_bikes_available_types.mechanical,num_bikes_available_types.ebike,num_docks_available,last_reported,is_charging_station,status,traffic,last_updated
0,1.0,15.0,10.0,5.0,30.0,1.735712e+09,True,IN_SERVICE,NaN,1.735712e+09
1,1.0,15.0,9.0,6.0,30.0,1.735713e+09,True,IN_SERVICE,NaN,1.735713e+09
2,1.0,12.0,9.0,3.0,33.0,1.735714e+09,True,IN_SERVICE,NaN,1.735714e+09
3,1.0,12.0,9.0,3.0,34.0,1.735715e+09,True,IN_SERVICE,NaN,1.735715e+09
4,1.0,12.0,9.0,3.0,34.0,1.735716e+09,True,IN_SERVICE,NaN,1.735716e+09


In [28]:
from pathlib import Path


path = Path("data/bicing_stations_status")
files = list(path.glob('*.parquet')) 

if not files:
    print("No files found in the specified directory.")
    database.disconnect()

In [29]:
insert_query = """
    INSERT INTO station_status_input (
        station_id, report_time, num_bikes, num_ebikes, num_mechanical, docks_free, status
    ) VALUES %s
    ON CONFLICT (station_id, report_time) DO NOTHING;
    """

In [33]:
existing_stations_query = "SELECT id FROM stations;"
result = database.execute_query(existing_stations_query, fetch=True)
valid_station_ids = {row['id'] for row in result}

Database connection established successfully.


In [31]:
database.connection.close()

In [34]:
import psycopg2

for file_path in files:
    print(f"Processing Parquet file: {file_path.name}...")
    
    df = pd.read_parquet(file_path)
    df['report_time'] = pd.to_datetime(df['last_reported'], unit='s', utc=True)

    column_mapping = {
        'station_id': 'station_id',
        'report_time': 'report_time',
        'num_bikes_available': 'num_bikes',
        'num_bikes_available_types.ebike': 'num_ebikes',
        'num_bikes_available_types.mechanical': 'num_mechanical',
        'num_docks_available': 'docks_free',
        'status': 'status'
    }

    df_clean = df[list(column_mapping.keys())].copy()
    df_clean = df_clean.dropna(subset=['station_id', 'report_time'])
    
    initial_count = len(df_clean)
    df_clean = df_clean[df_clean['station_id'].isin(valid_station_ids)]
    filtered_count = initial_count - len(df_clean)

    if filtered_count > 0:
        print(f"Filtered out {filtered_count} records with unknown station_ids.")

    df_clean = df_clean.fillna({
        'num_bikes_available': 0,
        'num_bikes_available_types.ebike': 0,
        'num_bikes_available_types.mechanical': 0,
        'num_docks_available': 0,
        'status': 'UNKNOWN'
    })

    data_tuples = [
        (
            int(row[0]), 
            row[1], 
            int(row[2]), 
            int(row[3]), 
            int(row[4]), 
            int(row[5]), 
            str(row[6])
        ) 
        for row in df_clean.to_numpy()
    ]

    if not data_tuples:
        print(f"No valid records to ingest for {file_path.name}")
        continue

    try:
        with database.connection.cursor() as cursor:
            psycopg2.extras.execute_values(
                cursor, insert_query, data_tuples, page_size=15000
            )
        database.connection.commit()
        print(f"Successfully ingested {len(data_tuples)} records from {file_path.name}")
    except Exception as e:
        database.connection.rollback()
        print(f"Failed to ingest {file_path.name}: {e}")

database.disconnect()

Processing Parquet file: 2025_02_Febrer_BicingNou_ESTACIONS_filtered.parquet...
Filtered out 10741 records with unknown station_ids.
Successfully ingested 908672 records from 2025_02_Febrer_BicingNou_ESTACIONS_filtered.parquet
Processing Parquet file: 2025_09_Setembre_BicingNou_ESTACIONS_filtered.parquet...
Filtered out 9492 records with unknown station_ids.
Successfully ingested 1032569 records from 2025_09_Setembre_BicingNou_ESTACIONS_filtered.parquet
Processing Parquet file: 2025_03_Marc_BicingNou_ESTACIONS_filtered.parquet...
Filtered out 11859 records with unknown station_ids.
Successfully ingested 1002704 records from 2025_03_Marc_BicingNou_ESTACIONS_filtered.parquet
Processing Parquet file: 2025_12_Desembre_BicingNou_ESTACIONS_filtered.parquet...
Filtered out 5351 records with unknown station_ids.
Successfully ingested 737258 records from 2025_12_Desembre_BicingNou_ESTACIONS_filtered.parquet
Processing Parquet file: 2025_01_Gener_BicingNou_ESTACIONS_filtered.parquet...
Filtered 

In [35]:
station_56 = stations[stations['station_id'] == 56]

In [36]:
station_56.head()

,station_id,name,lat,lon,capacity


### Migrate to mdb table

In [38]:
migration_query = """
    INSERT INTO station_status_mdb (
        station_id,
        bikes_history,
        ebikes_history,
        mechanical_history,
        docks_history,
        status_history
    )
    SELECT
        station_id,
        tint_seq(array_agg(tint_inst(num_bikes, report_time) ORDER BY report_time)),
        tint_seq(array_agg(tint_inst(num_ebikes, report_time) ORDER BY report_time)),
        tint_seq(array_agg(tint_inst(num_mechanical, report_time) ORDER BY report_time)),
        tint_seq(array_agg(tint_inst(docks_free, report_time) ORDER BY report_time)),
        ttext_seq(array_agg(ttext_inst(status, report_time) ORDER BY report_time))
    FROM station_status_input
    GROUP BY station_id
    ON CONFLICT (station_id) DO UPDATE SET
        bikes_history = EXCLUDED.bikes_history,
        ebikes_history = EXCLUDED.ebikes_history,
        mechanical_history = EXCLUDED.mechanical_history,
        docks_history = EXCLUDED.docks_history,
        status_history = EXCLUDED.status_history;
"""

In [39]:
try:
    database.connect()
    print("Starting migration to MobilityDB analytic layer...")
    database.execute_query(migration_query, fetch=False)
    print("Successfully migrated data to station_status_mdb")
except Exception as e:
    print(f"Migration failed: {e}")
finally:
    database.disconnect()

Database connection established successfully.
Starting migration to MobilityDB analytic layer...
Successfully migrated data to station_status_mdb
Database connection closed.
